# importy i funckje

In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import pandas_ta as ta
from tqdm import tqdm
import numpy as np

# widgety do wyboru ustawień (interwal, filtry)
from core.poczatek_ustawienia import create_settings_ui

# kontrola taliba
from core.kontrola_taliba import create_talib_control

# zaladowanie df do pamieci
from core.ladowanie_danych import create_stock_dfs, add_indicators

# dodanie formacji
from core.dodanie_formacji import add_candle_patterns

# giga plot
from core.main_plot import create_chart_ui


In [13]:
# Indicator functions from pandas-ta
def hma(close, length):
    return ta.hma(close, length=length)

def rsi(close, length):
    return ta.rsi(close, length=length)

def aroon(high, low, length):
    result = ta.aroon(high, low, length=length)
    # Return just the Aroon Up indicator
    aroon_cols = [col for col in result.columns if 'AROONU' in col]
    return result[aroon_cols[0]] if aroon_cols else pd.Series(0, index=high.index)

def williams_r(high, low, close, length):
    return ta.willr(high, low, close, length=length)

def supertrend(df):
    result = ta.supertrend(df['High'], df['Low'], df['Close'], length=10, multiplier=3)
    supertrend_cols = [col for col in result.columns if 'SUPERT' in col]
    return result[supertrend_cols[0]] if supertrend_cols else pd.Series(0, index=df.index)


# poczatkowe ustawienia

In [14]:
settings_panel, settings = create_settings_ui()
display(settings_panel)

In [15]:
print("Wybrane ustawienia:")
for key, value in settings.items():
    print(f"{key}: {value}")

Wybrane ustawienia:
market: stocks
interval: 1week
week_start: MON
vol_enabled: True
vol_ratio_window: 20
vol_ratio_threshold: 1.2000000000000002
cmo_enabled: True
cmo_len: 6
cmo_thres: -35
cmo_thres_prev: -50


# kontrola taliba

In [16]:
talib_panel, apply_params_fn, registry = create_talib_control(settings)
display(talib_panel)
apply_params_fn()


# zaladowanie df'ow

In [17]:
dfs_1d, dfs_1w = create_stock_dfs(settings)

Loading data:   0%|          | 0/479 [00:00<?, ?it/s]

In [18]:
ex_1d = dfs_1d.get("ALLE")
ex_1w = dfs_1w.get("ISRG")

# plot

In [19]:
settings

{'market': 'stocks',
 'interval': '1week',
 'week_start': 'MON',
 'vol_enabled': True,
 'vol_ratio_window': 20,
 'vol_ratio_threshold': 1.2000000000000002,
 'cmo_enabled': True,
 'cmo_len': 6,
 'cmo_thres': -35,
 'cmo_thres_prev': -50}

In [20]:
symbols = list(dfs_1d.keys())
ui = create_chart_ui(dfs_1d, dfs_1w, settings, symbols, 
                      add_indicators, add_candle_patterns,
                      hma, rsi, aroon, williams_r, supertrend)
display(ui)

In [27]:
df[bear_cols].describe()


,shooting_star,hanging_man,dark_cloud_cover,evening_star
count,263.000000,263.000000,263.0,263.000000
mean,-1.140684,-1.901141,0.0,-0.380228
std,10.639440,13.682528,0.0,6.166264
min,-100.000000,-100.000000,0.0,-100.000000
25%,0.000000,0.000000,0.0,0.000000
50%,0.000000,0.000000,0.0,0.000000
75%,0.000000,0.000000,0.0,0.000000
max,0.000000,0.000000,0.0,0.000000


In [31]:
import os
import glob
import numpy as np


signal_cols = ['hammer', 'inverted_hammer', 'engulfing', 'piercing_line']

bear_cols = ['shooting_star', 'hanging_man', 'dark_cloud_cover', 'evening_star']

signal_filters = {'CMO': -20, 'vol_ratio': 1.0}

bull_filters = {'CMO': -35, 'vol_ratio': 1.0}
bear_filters = {'CMO': 35, 'vol_ratio': 0.0}

ohlcv_cols = ['Open', 'High', 'Low', 'Close', 'Volume']

n = 5  # liczba ostatnich świec do sprawdzenia
output_dir = "git/data"

for interval in ['1d', '1w']:

    # Clean directory for this interval
    files_to_remove = glob.glob(f"{output_dir}/*_{interval}.csv")
    for file in files_to_remove:
        os.remove(file)

    for symbol in tqdm(symbols, desc=f"Processing {interval}"):

        df = dfs_1d.get(symbol) if interval == '1d' else dfs_1w.get(symbol)

        if df is None:
            continue

        df = add_candle_patterns(df, settings)

        # ----------------------------------
        # SIGNAL (general)
        # ----------------------------------
        mask_any = df[signal_cols] != 0

        df['signal'] = [
            [pattern for pattern, flag in zip(signal_cols, row) if flag]
            for row in mask_any.values
        ]

        # ----------------------------------
        # BULLISH PATTERNS (value > 0)
        # ----------------------------------
        bull_condition = (
            (df['CMO'] < bull_filters['CMO']) &
            (df['vol_ratio'] > bull_filters['vol_ratio'])
        )

        df['bull_patterns'] = [
            [pattern for pattern, val in zip(signal_cols, row) if val > 0]
            if bull_condition.iloc[i] else []
            for i, row in enumerate(df[signal_cols].values)
        ]

        # ----------------------------------
        # BEARISH PATTERNS (value < 0)
        # ----------------------------------
        bear_condition = (
            (df['CMO'] > bear_filters['CMO']) &
            (df['vol_ratio'] > bear_filters['vol_ratio'])
        )

        df['bear_patterns'] = [
            [pattern for pattern, val in zip(bear_cols, row) if val < 0]
            if bear_condition.iloc[i] else []
            for i, row in enumerate(df[bear_cols].values)
        ]

        # ----------------------------------
        # CHECK LAST N CANDLES
        # ----------------------------------
        last_n = df.tail(n)

        has_recent_signal = any(
            (len(row['bull_patterns']) > 0) 
            #or
            #(len(row['bear_patterns']) > 0)
            for _, row in last_n.iterrows()
        )

        if has_recent_signal:
            df_reset = df.reset_index()

            cols_to_save = (
                ['Date'] +
                ohlcv_cols +
                ['signal', 'bull_patterns', 'bear_patterns']
            )

            df_to_save = df_reset[cols_to_save].tail(200)

            df_to_save.to_csv(
                f"{output_dir}/{symbol}_{interval}.csv",
                index=False
            )

Processing 1w: 100%|██████████| 479/479 [00:12<00:00, 39.15it/s]


In [ ]:
eee

# hehe statystyka

## ladny print

In [32]:
def print_trade_results_stats(trades_df):
    if trades_df is None or trades_df.empty:
        print("❌ No trades to analyze.")
        return

    df = trades_df.copy()

    total_trades = len(df)
    wins = (df["return_pct"] > 0).sum()
    losses = (df["return_pct"] <= 0).sum()
    win_rate = wins / total_trades * 100 if total_trades else 0

    print("=" * 80)
    print("FIRST PROFIT STRATEGY — OVERALL PERFORMANCE")
    print("=" * 80)

    print(f"Trades        : {total_trades}")
    print(f"Wins / Losses : {wins} / {losses}")
    print(f"Win rate      : {win_rate:.2f}%")
    print(f"Avg return    : {df['return_pct'].mean():.2f}%")
    print(f"Median return : {df['return_pct'].median():.2f}%")

    if "hold_bars" in df:
        print(f"Avg hold      : {df['hold_bars'].mean():.2f} bars")

    # ======================================================================
    # EXIT REASONS
    # ======================================================================

    if "exit_reason" in df:
        print("\n" + "=" * 80)
        print("EXIT REASONS")
        print("=" * 80)

        exit_stats = df.groupby("exit_reason").agg(
            trades=("return_pct", "count"),
            win_rate=("return_pct", lambda x: (x > 0).mean() * 100),
            avg_return_pct=("return_pct", "mean"),
        )

        exit_stats["occurrence_pct"] = (
            exit_stats["trades"] / total_trades * 100
        )

        print(exit_stats.sort_values("trades", ascending=False).round(2))

    # ======================================================================
    # PATTERN PERFORMANCE
    # ======================================================================

    if "pattern" in df:
        print("\n" + "=" * 80)
        print("PATTERN PERFORMANCE")
        print("=" * 80)

        pattern_stats = df.groupby("pattern").agg(
            trades=("return_pct", "count"),
            win_rate=("return_pct", lambda x: (x > 0).mean() * 100),
            avg_return_pct=("return_pct", "mean"),
        )

        pattern_stats["occurrence_pct"] = (
            pattern_stats["trades"] / total_trades * 100
        )

        print(
            pattern_stats
            .sort_values("avg_return_pct", ascending=False)
            .round(2)
        )

    # ======================================================================
    # DURATION METRICS
    # ======================================================================

    if "hold_bars" in df:
        print("\n" + "=" * 80)
        print("TRADE DURATION")
        print("=" * 80)

        print(f"Avg hold (bars)    : {df['hold_bars'].mean():.2f}")
        print(f"Median hold (bars) : {df['hold_bars'].median():.0f}")
        print(f"Max hold (bars)    : {df['hold_bars'].max()}")

        #if "hold_time" in df and df["hold_time"].notna().any():
            #print(f"Avg hold (time)    : {df['hold_time'].mean()}")

        print("\nHold bars by exit reason:")
        print(
            df.groupby("exit_reason")["hold_bars"]
            .agg(["mean", "median", "max"])
            .round(2)
        )

    # ======================================================================
    # RISK METRICS
    # ======================================================================

    print("\n" + "=" * 80)
    print("RISK METRICS")
    print("=" * 80)

    avg_win = df.loc[df["return_pct"] > 0, "return_pct"].mean()
    avg_loss = df.loc[df["return_pct"] <= 0, "return_pct"].mean()

    profit_factor = (
        df.loc[df["return_pct"] > 0, "return_pct"].sum()
        / abs(df.loc[df["return_pct"] <= 0, "return_pct"].sum())
        if losses > 0 else float("inf")
    )

    expectancy = (
        (wins / total_trades) * avg_win +
        (losses / total_trades) * avg_loss
    )

    print(f"Avg win        : {avg_win:.2f}%")
    print(f"Avg loss       : {avg_loss:.2f}%")
    print(f"Profit factor  : {profit_factor:.2f}")
    print(f"Expectancy     : {expectancy:.2f}%")


## patterns_df

In [33]:
def extract_patterns_by_cols(dfs_dict, cols, desc="Scanning for all patterns"):
    rows = []

    for symbol, df in tqdm(dfs_dict.items(), desc=desc):
        if df is None or df.empty:
            continue
        
        df = add_candle_patterns(df, settings)

        for col in cols:
            if col not in df.columns:
                continue

            series = pd.to_numeric(df[col], errors="coerce")
            hits = series[series > 0]

            for idx, val in hits.items():
                rows.append(
                    {
                        "symbol": symbol,
                        "column": col,
                        "timestamp": idx,
                        "value": val,
                    }
                )

    return pd.DataFrame(rows)

In [34]:
pattern_cols = ['hammer', 'inverted_hammer', 'engulfing_bull', 'piercing_line']

In [35]:
patterns_df_1d = extract_patterns_by_cols(
    dfs_1d,
    pattern_cols,
    desc="🔍 Scanning 1D patterns",
)

patterns_df_1w = extract_patterns_by_cols(
    dfs_1w,
    pattern_cols,
    desc="🔍 Scanning 1W patterns",
)


🔍 Scanning 1W patterns: 100%|██████████| 479/479 [00:10<00:00, 45.43it/s]


In [36]:
patterns_df_1d

,symbol,column,timestamp,value
0,MMM,hammer,2021-03-22,100.0
1,MMM,hammer,2021-04-19,100.0
2,MMM,hammer,2021-05-19,100.0
3,MMM,hammer,2021-05-26,100.0
4,MMM,hammer,2021-10-18,100.0
...,...,...,...,...
59700,ZTS,engulfing_bull,2026-01-05,1.0
59701,ZTS,engulfing_bull,2026-01-30,1.0
59702,ZTS,engulfing_bull,2026-02-20,1.0
59703,ZTS,piercing_line,2022-10-21,100.0


## first profit paczymy

In [37]:
def first_profit_strat(
    max_hold=10,

    hard_tp=0.02,
    hard_sl=0.08,
    vol_tp_mult=1.0,
    vol_sl_mult=3.0,
    min_profit_exit=0.005,

    use_hard_tp=True,
    use_hard_sl=True,
    use_vol_tp=False,
    use_vol_sl=False,

    require_vol_confirmation=True,
    require_cmo_confirmation=True,

    interval="1d",
    weekly_exit_on_daily=True,
    entry_offset=1,

    pattern_cols=None,
    debug=True,
):
    global patterns_df_1d, patterns_df_1w, dfs_1d, dfs_1w

    entry_offset = max(entry_offset, 1)  # prevent same-candle entry

    patterns = patterns_df_1d if interval == "1d" else patterns_df_1w
    entry_dfs = dfs_1d if interval == "1d" else dfs_1w
    exit_dfs = dfs_1d if (interval == "1w" and weekly_exit_on_daily) else entry_dfs

    patterns = patterns.sort_values("timestamp")

    trades = []

    debug_counts = {
        "no_pattern": 0,
        "symbol_missing": 0,
        "timestamp_missing": 0,
        "vol_reject": 0,
        "cmo_reject": 0,
        "entry_oob": 0,
        "exit_df_missing": 0,
        "future_empty": 0,
    }

    if pattern_cols is None:
        pattern_cols = patterns["column"].unique().tolist()

    for _, row in tqdm(
        patterns.iterrows(),
        total=len(patterns),
        desc="📊 Building trades",
    ):
        symbol = row["symbol"]
        ts = row["timestamp"]
        pattern = row["column"]

        # ===== PATTERN FILTER =====
        if pattern not in pattern_cols:
            debug_counts["no_pattern"] += 1
            continue

        # ===== SYMBOL =====
        if symbol not in entry_dfs:
            debug_counts["symbol_missing"] += 1
            continue

        entry_df = entry_dfs[symbol]
        if ts not in entry_df.index:
            debug_counts["timestamp_missing"] += 1
            continue

        sig = entry_df.loc[ts]

        # ===== CONFIRMATIONS =====
        if require_vol_confirmation:
            v = sig.get("VOL_SIGNIFICANT", 0)
            if isinstance(v, pd.Series):
                v = v.iloc[0]
            if v != 1:
                debug_counts["vol_reject"] += 1
                continue

        if require_cmo_confirmation:
            c = sig.get("DOWNTREND_SHORT", 0)
            if isinstance(c, pd.Series):
                c = c.iloc[0]
            if c != 1:
                debug_counts["cmo_reject"] += 1
                continue

        # ===== ENTRY =====
        entry_loc = entry_df.index.get_loc(ts) + entry_offset
        if entry_loc >= len(entry_df):
            debug_counts["entry_oob"] += 1
            continue

        entry_time = entry_df.index[entry_loc]
        entry_row = entry_df.iloc[entry_loc]
        entry_price = entry_row["Open"]
        atr = entry_row.get("ATR", None)

        # ===== EXIT DATA =====
        exit_df = exit_dfs.get(symbol)
        if exit_df is None:
            debug_counts["exit_df_missing"] += 1
            continue

        future = exit_df.loc[exit_df.index > entry_time].iloc[:max_hold]
        if future.empty:
            debug_counts["future_empty"] += 1
            continue

        # ===== TP / SL =====
        tp = sl = None

        if use_hard_tp:
            tp = entry_price * (1 + hard_tp)
        if use_hard_sl:
            sl = entry_price * (1 - hard_sl)

        if atr is not None:
            if use_vol_tp:
                tp = entry_price + atr * vol_tp_mult
            if use_vol_sl:
                sl = entry_price - atr * vol_sl_mult

        exit_price = future.iloc[-1]["Close"]
        exit_time = future.index[-1]
        exit_reason = "TIME_EXIT"
        exit_loc = future.index.get_loc(exit_time)

        highs = future["High"].values
        lows = future["Low"].values
        closes = future["Close"].values

        # ===== SL FIRST =====
        if sl is not None:
            hit = np.where(lows <= sl)[0]
            if len(hit):
                exit_loc = hit[0]
                exit_price = sl
                exit_time = future.index[exit_loc]
                exit_reason = "SL"

        # ===== TP =====
        if exit_reason == "TIME_EXIT" and tp is not None:
            hit = np.where(highs >= tp)[0]
            if len(hit):
                exit_loc = hit[0]
                exit_price = tp
                exit_time = future.index[exit_loc]
                exit_reason = "TP"

        # ===== MIN PROFIT =====
        if exit_reason == "TIME_EXIT" and min_profit_exit and min_profit_exit > 0:
            profits = (closes - entry_price) / entry_price
            hit = np.where(profits >= min_profit_exit)[0]
            if len(hit):
                exit_loc = hit[0]
                exit_price = closes[exit_loc]
                exit_time = future.index[exit_loc]
                exit_reason = "MIN_PROFIT"

        # ===== METRICS =====
        ret_pct = (exit_price - entry_price) / entry_price * 100
        hold_bars = exit_loc + 1

        hold_time = None
        if isinstance(entry_time, pd.Timestamp) and isinstance(exit_time, pd.Timestamp):
            hold_time = exit_time - entry_time

        trades.append({
            "symbol": symbol,
            "pattern": pattern,
            "entry_time": entry_time,
            "exit_time": exit_time,
            "entry_price": entry_price,
            "exit_price": exit_price,
            "return_pct": ret_pct,
            "exit_reason": exit_reason,
            "hold_bars": hold_bars,
            "hold_time": hold_time,
        })

    trades_df = pd.DataFrame(trades)

    if trades_df.empty and debug:
        print("⚠️ No trades generated. Debug summary:")
        for k, v in debug_counts.items():
            print(f"  {k}: {v}")

    return trades_df


In [68]:
trades_df = first_profit_strat(
    max_hold=15,

    hard_tp=0.05,
    hard_sl=0.05,

    vol_tp_mult=0.5,
    vol_sl_mult=1.0,
    
    min_profit_exit=0.005,

    use_hard_tp=True,
    use_hard_sl=True,
    use_vol_tp=False,
    use_vol_sl=False,

    require_vol_confirmation=True,
    require_cmo_confirmation=True,

    interval='1w',
    weekly_exit_on_daily=True,
    entry_offset=0,

    pattern_cols=[
        'hammer',
        'inverted_hammer',
        #'engulfing_bull',
        #'piercing_line',
        ],  
)


📊 Building trades: 100%|██████████| 12421/12421 [00:01<00:00, 10351.29it/s]


In [70]:
trades_df

,symbol,pattern,entry_time,exit_time,entry_price,exit_price,return_pct,exit_reason,hold_bars,hold_time
0,LVS,inverted_hammer,2021-08-01,2021-08-02,45.340000,43.073000,-5.000000,SL,1,1 days
1,WYNN,inverted_hammer,2021-08-01,2021-08-03,102.639999,97.507999,-5.000000,SL,2,2 days
2,WYNN,hammer,2021-08-15,2021-08-17,96.699997,91.864997,-5.000000,SL,2,2 days
3,UBER,hammer,2021-08-15,2021-08-16,44.189999,41.980499,-5.000000,SL,1,1 days
4,MTCH,hammer,2021-08-29,2021-08-30,133.500000,140.175000,5.000000,TP,1,1 days
...,...,...,...,...,...,...,...,...,...,...
619,TRMB,hammer,2026-02-15,2026-02-18,65.779999,66.910004,1.717855,MIN_PROFIT,2,3 days
620,CDNS,hammer,2026-02-15,2026-02-18,283.489990,297.664490,5.000000,TP,2,3 days
621,EFX,hammer,2026-02-15,2026-02-20,194.520004,197.460007,1.511414,MIN_PROFIT,4,5 days
622,PAYC,hammer,2026-02-15,2026-02-17,130.729996,124.193496,-5.000000,SL,1,2 days


## wyniki

In [71]:
print_trade_results_stats(trades_df)

FIRST PROFIT STRATEGY — OVERALL PERFORMANCE
Trades        : 624
Wins / Losses : 323 / 301
Win rate      : 51.76%
Avg return    : -0.04%
Median return : 0.91%
Avg hold      : 4.93 bars

EXIT REASONS
             trades  win_rate  avg_return_pct  occurrence_pct
exit_reason                                                  
SL              299       0.0           -5.00           47.92
TP              283     100.0            5.00           45.35
MIN_PROFIT       40     100.0            1.47            6.41
TIME_EXIT         2       0.0           -1.12            0.32

PATTERN PERFORMANCE
                 trades  win_rate  avg_return_pct  occurrence_pct
pattern                                                          
inverted_hammer     181     56.91            0.52           29.01
hammer              443     49.66           -0.26           70.99

TRADE DURATION
Avg hold (bars)    : 4.93
Median hold (bars) : 3
Max hold (bars)    : 15

Hold bars by exit reason:
              mean  median  m

In [64]:
import numpy as np
import pandas as pd
from itertools import product

# Define your grid
hard_tp_vals = np.arange(0.02, 0.06, 0.01)   # 2% to 10%
hard_sl_vals = np.arange(0.02, 0.06, 0.01)  # 1% to 5%
hold_vals    = range(2, 6)                  # 2 to 10 bars

param_grid = list(product(hard_tp_vals, hard_sl_vals, hold_vals))
total_iters = len(param_grid)

results = []

for tp, sl, hold in tqdm(param_grid, total=total_iters, desc="Grid Search"):
     
    trades_df = first_profit_strat(
        max_hold=hold,

        hard_tp=tp,
        hard_sl=sl,

        vol_tp_mult=0.5,
        vol_sl_mult=1.0,
        
        min_profit_exit=0.005,

        use_hard_tp=True,
        use_hard_sl=True,
        use_vol_tp=False,
        use_vol_sl=False,

        require_vol_confirmation=True,
        require_cmo_confirmation=True,

        interval='1d',
        weekly_exit_on_daily=True,
        entry_offset=0,

        pattern_cols=[
            'hammer',
            'inverted_hammer',
        ],
    )

    if len(trades_df) == 0:
        continue

    avg_return = trades_df['return_pct'].mean()

    results.append({
        'hard_tp': tp,
        'hard_sl': sl,
        'max_hold': hold,
        'avg_return': avg_return,
        'num_trades': len(trades_df)
    })

results_df = pd.DataFrame(results)

# Sort by best average return
results_df = results_df.sort_values(by='avg_return', ascending=False)

# Show top 10
print(results_df.head(10))

Grid Search: 100%|██████████| 64/64 [07:31<00:00,  7.06s/it]

    hard_tp  hard_sl  max_hold  avg_return  num_trades
63     0.05     0.05         5    0.344044        3436
62     0.05     0.05         4    0.342892        3436
48     0.05     0.02         2    0.317105        3436
61     0.05     0.05         3    0.302750        3436
49     0.05     0.02         3    0.300067        3436
46     0.04     0.05         4    0.285034        3436
32     0.04     0.02         2    0.280410        3436
58     0.05     0.04         4    0.258548        3436
45     0.04     0.05         3    0.258319        3436
47     0.04     0.05         5    0.255435        3436
